# Length-only human preference predictor: offline Kaggle CPU submission
**Status:** prepared from reproducible repository code; this Kaggle Notebook has **not** been committed or scored on the leaderboard yet.

Attach the official `llm-classification-finetuning` competition data via **Add Input** after joining and accepting rules. Set **Internet = Off**; no GPU, model downloads, API token or external dependency installation is needed.

This experiment fits a numeric character-length-only baseline with regularization C=10.0, chosen using an inner fold from an earlier 12K-row exploratory pilot. Its held-out pilot log loss was **1.057783**; results from this new full-training Notebook or Kaggle hidden test are **not yet known**. The held-out pilot had a row-random split (repeated prompts may leak) and was repeatedly used for exploratory comparisons.

**Research limits:** Character length is correlated with many qualities. The model cannot identify warmth, user intent or causal effects. It is designed to produce a robust offline competition submission, not to make substantive claims about human psychology.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
root = Path('/kaggle/input/llm-classification-finetuning')
assert (root / 'train.csv').is_file(), 'Attach official competition data via Add Input'
assert (root / 'test.csv').is_file(), 'Attach official competition data via Add Input'
print('scikit-learn version:', sklearn.__version__)


In [ ]:
# Generated by python scripts/sync_length_notebook.py; no internet needed on Kaggle.
from pathlib import Path
import sys
source_dir = Path('/kaggle/working/src')
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / '__init__.py').write_text('', encoding='utf-8')
(source_dir / 'baseline.py').write_text("\"\"\"Leakage-controlled, swap-augmented TF-IDF baseline for Kaggle LLM preference prediction.\n\nThis is a classical ML baseline, not an LLM fine-tuning run.\n\"\"\"\nimport argparse\nimport ast\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack, vstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nTARGETS = [\"winner_model_a\", \"winner_model_b\", \"winner_tie\"]\nTEXT_COLUMNS = [\"prompt\", \"response_a\", \"response_b\"]\n\n\ndef flatten_messages(value, max_chars=2400):\n    \"\"\"Normalize Kaggle's serialized lists of turns; cap length for a CPU starter.\"\"\"\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return \"\"\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith(\"[\"):\n            try:\n                value = json.loads(text)\n            except (ValueError, TypeError):\n                try:\n                    value = ast.literal_eval(text)\n                except (ValueError, SyntaxError):\n                    value = text\n        else:\n            value = text\n    if isinstance(value, (list, tuple)):\n        text = \" \".join(\"\" if item is None else str(item) for item in value)\n    else:\n        text = str(value)\n    return text[:max_chars]\n\n\ndef normalized_frame(df):\n    missing = [c for c in TEXT_COLUMNS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing text columns: {missing}\")\n    return pd.DataFrame(\n        {col: [flatten_messages(v) for v in df[col]] for col in TEXT_COLUMNS},\n        index=df.index,\n    )\n\n\ndef get_labels(df):\n    missing = [c for c in TARGETS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing label columns: {missing}\")\n    y = df[TARGETS].to_numpy(dtype=int)\n    if not np.all(y.sum(axis=1) == 1) or not np.all((y == 0) | (y == 1)):\n        raise ValueError(\"Expected exactly one binary winner label per training row\")\n    return y.argmax(axis=1)\n\n\ndef flip_pairs(df):\n    flipped = df.copy()\n    flipped[\"response_a\"], flipped[\"response_b\"] = (\n        df[\"response_b\"].copy(), df[\"response_a\"].copy()\n    )\n    return flipped\n\n\ndef make_vectorizer(df):\n    # Only fit on training-partition texts; do not fit on held-out validation/test.\n    min_df = 2 if len(df) >= 30 else 1\n    vectorizer = TfidfVectorizer(\n        ngram_range=(1, 2), max_features=35000, min_df=min_df,\n        strip_accents=\"unicode\", sublinear_tf=True, dtype=np.float32,\n    )\n    vectorizer.fit(\n        df[\"prompt\"].tolist() + df[\"response_a\"].tolist() +\n        df[\"response_b\"].tolist()\n    )\n    return vectorizer\n\n\ndef pair_features(df, vectorizer):\n    q = vectorizer.transform(df[\"prompt\"])\n    a = vectorizer.transform(df[\"response_a\"])\n    b = vectorizer.transform(df[\"response_b\"])\n    len_a = df[\"response_a\"].str.len().to_numpy(dtype=np.float32)\n    len_b = df[\"response_b\"].str.len().to_numpy(dtype=np.float32)\n    len_q = df[\"prompt\"].str.len().to_numpy(dtype=np.float32)\n    numeric = np.column_stack([\n        np.log1p(len_a) - np.log1p(len_b),\n        (np.log1p(len_a) + np.log1p(len_b)) / 2,\n        np.log1p(len_q),\n    ]) / 10.0\n    return hstack([q, a - b, (a + b) * 0.5, csr_matrix(numeric)],\n                  format=\"csr\", dtype=np.float32)\n\n\ndef fit_baseline(df, y):\n    vectorizer = make_vectorizer(df)\n    x_original = pair_features(df, vectorizer)\n    x_flipped = pair_features(flip_pairs(df), vectorizer)\n    swapped_labels = np.where(y == 0, 1, np.where(y == 1, 0, 2))\n    model = LogisticRegression(C=2.0, max_iter=300, random_state=42)\n    model.fit(vstack([x_original, x_flipped], format=\"csr\"),\n              np.concatenate([y, swapped_labels]))\n    return {\"vectorizer\": vectorizer, \"model\": model, \"targets\": TARGETS}\n\n\ndef predict_prob(bundle, df):\n    features = pair_features(df, bundle[\"vectorizer\"])\n    raw = bundle[\"model\"].predict_proba(features)\n    out = np.zeros((len(df), len(TARGETS)), dtype=np.float64)\n    for col_idx, class_idx in enumerate(bundle[\"model\"].classes_):\n        out[:, int(class_idx)] = raw[:, col_idx]\n    return out / out.sum(axis=1, keepdims=True)\n\n\ndef train(train_csv, out_dir, validation_fraction=0.15):\n    raw = pd.read_csv(train_csv)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if len(np.unique(y)) != 3 or np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class must have at least two examples for validation\")\n    x_tr, x_val, y_tr, y_val = train_test_split(\n        df, y, test_size=validation_fraction, random_state=42, stratify=y\n    )\n    validation_bundle = fit_baseline(x_tr, y_tr)\n    val_probs = predict_prob(validation_bundle, x_val)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_probs, labels=[0, 1, 2])),\n        \"train_rows\": int(len(x_tr)),\n        \"validation_rows\": int(len(x_val)),\n        \"full_rows\": int(len(df)),\n        \"seed\": 42,\n        \"note\": \"Random stratified split; not a competition leaderboard result.\",\n    }\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \"validation_metrics.json\").write_text(\n        json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    # Refit using all labeled data only after holding out validation above.\n    joblib.dump(fit_baseline(df, y), output / \"baseline.joblib\")\n    return metrics\n\n\ndef predict(test_csv, model_path, out_csv):\n    test = pd.read_csv(test_csv)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Test CSV must contain id\")\n    bundle = joblib.load(model_path)  # Load only artifacts you created/trust.\n    probabilities = predict_prob(bundle, normalized_frame(test))\n    result = pd.DataFrame(probabilities, columns=TARGETS)\n    result.insert(0, \"id\", test[\"id\"])\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out_csv, index=False)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\"command\", required=True)\n    fit = sub.add_parser(\"train\")\n    fit.add_argument(\"--train\", default=\"data/train.csv\")\n    fit.add_argument(\"--out\", default=\"artifacts\")\n    infer = sub.add_parser(\"predict\")\n    infer.add_argument(\"--test\", default=\"data/test.csv\")\n    infer.add_argument(\"--model\", default=\"artifacts/baseline.joblib\")\n    infer.add_argument(\"--out\", default=\"submission.csv\")\n    args = parser.parse_args()\n    if args.command == \"train\":\n        print(json.dumps(train(args.train, args.out), indent=2))\n    else:\n        print(f\"Wrote {len(predict(args.test, args.model, args.out))} rows: {args.out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
(source_dir / 'length_baseline.py').write_text("\"\"\"Nested-tuned, length-only A/B/tie baseline, exploratory validation.\n\nThis tests whether simple observable character lengths provide predictive signal.\nIt neither measures warmth nor establishes a causal preference for verbosity.\nOnly aggregate JSON may be published; official Kaggle data stay local.\n\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with open(path, 'rb') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b''):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef feature_matrix(df):\n    \"\"\"No semantic word features: lengths of complete joined conversation fields.\"\"\"\n    def lengths(col):\n        return np.asarray(\n            [len(flatten_messages(value, max_chars=2_000_000)) for value in df[col]],\n            dtype=np.float64,\n        )\n    a = np.log1p(lengths(\"response_a\"))\n    b = np.log1p(lengths(\"response_b\"))\n    q = np.log1p(lengths(\"prompt\"))\n    delta = a - b\n    return np.column_stack([\n        delta,\n        np.abs(delta),\n        (a + b) / 2,\n        q,\n        delta * (q / 10),\n        (a - b) / np.maximum((a + b), 1.0),\n    ])\n\n\ndef fit_length_model(frame, labels, c=1.0):\n    original = feature_matrix(frame)\n    swapped = feature_matrix(flip_pairs(frame))\n    swapped_labels = np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n    model = LogisticRegression(C=c, max_iter=700, random_state=42)\n    model.fit(\n        np.vstack([original, swapped]),\n        np.concatenate([labels, swapped_labels])\n    )\n    return model\n\n\ndef predict_length_model(model, frame):\n    raw = model.predict_proba(feature_matrix(frame))\n    out = np.zeros((len(frame), 3), dtype=np.float64)\n    for idx, label in enumerate(model.classes_):\n        out[:, int(label)] = raw[:, idx]\n    return out\n\n\ndef logloss_delta_interval(labels, prediction, reference, draws=1000, seed=42):\n    \"\"\"Bootstrap per-row excess log loss; negative means model lower loss.\"\"\"\n    y = np.asarray(labels, dtype=np.int64)\n    selected_model = np.clip(prediction[np.arange(len(y)), y], 1e-15, 1)\n    selected_reference = np.clip(reference[np.arange(len(y)), y], 1e-15, 1)\n    delta = -np.log(selected_model) + np.log(selected_reference)\n    rng = np.random.default_rng(seed)\n    draws_arr = np.array([\n        delta[rng.integers(0, len(delta), len(delta))].mean()\n        for _ in range(draws)\n    ])\n    return [float(x) for x in np.quantile(draws_arr, [0.025, 0.975])]\n\n\ndef run_pilot(train_csv, out_dir, sample_size=12000, seed=42):\n    source = Path(train_csv)\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = pd.read_csv(source)\n    labels = get_labels(raw)\n    if len(raw) < 100 or np.min(np.bincount(labels, minlength=3)) < 10:\n        raise ValueError(\"Need >=100 labeled examples and >=10 per class\")\n    if sample_size < 0 or (sample_size > 0 and sample_size < 100):\n        raise ValueError(\"Pilot size should be 0 for all rows or >=100\")\n    if sample_size and sample_size < len(raw):\n        chosen, _ = train_test_split(\n            np.arange(len(raw)), train_size=sample_size,\n            stratify=labels, random_state=seed,\n        )\n        pilot = raw.iloc[chosen].reset_index(drop=True)\n    else:\n        pilot = raw.reset_index(drop=True)\n    pilot_y = get_labels(pilot)\n    # Exactly the outer split used in the 2026-09-23 TF-IDF benchmark.\n    outer_train, outer_val, y_train, y_val = train_test_split(\n        pilot, pilot_y, test_size=0.15, random_state=42, stratify=pilot_y\n    )\n    # Hyperparameter selection must be based only on an INNER calibration fold.\n    inner_train, inner_val, inner_y, inner_val_y = train_test_split(\n        outer_train, y_train, test_size=0.20,\n        random_state=1337, stratify=y_train\n    )\n    candidate_c = [0.01, 0.1, 1.0, 10.0]\n    inner_results = {}\n    for c in candidate_c:\n        model = fit_length_model(inner_train, inner_y, c=c)\n        inner_results[str(c)] = float(\n            log_loss(inner_val_y, predict_length_model(model, inner_val),\n                     labels=[0, 1, 2])\n        )\n    best_c = min(candidate_c, key=lambda c: inner_results[str(c)])\n    model = fit_length_model(outer_train, y_train, c=best_c)\n    prediction = predict_length_model(model, outer_val)\n    uniform = np.full_like(prediction, 1 / 3)\n    prior = np.bincount(y_train, minlength=3).astype(np.float64)\n    prior /= prior.sum()\n    prior_probs = np.tile(prior, (len(y_val), 1))\n    metrics = {\n        \"source\":\"Official Kaggle LLM Classification Finetuning training CSV\",\n        \"source_file_sha256\":file_sha256(source),\n        \"official_training_rows\":int(len(raw)),\n        \"pilot_rows\":int(len(pilot)),\n        \"outer_validation_rows\":int(len(y_val)),\n        \"outer_validation_log_loss_length_only\":float(log_loss(y_val, prediction,labels=[0,1,2])),\n        \"outer_validation_log_loss_training_prior\":float(log_loss(y_val, prior_probs,labels=[0,1,2])),\n        \"outer_validation_log_loss_uniform\":float(log_loss(y_val, uniform,labels=[0,1,2])),\n        \"length_minus_prior_log_loss_bootstrap_95pct\":logloss_delta_interval(\n            y_val, prediction, prior_probs, seed=seed,\n        ),\n        \"inner_validation_log_loss_by_c\":inner_results,\n        \"selected_c\":best_c,\n        \"outer_seed\":42,\n        \"inner_seed\":1337,\n        \"pilot_seed\":seed,\n        \"note\":\"EXPLORATORY comparison. Outer fold overlaps initial TF-IDF benchmark; do not reuse it indefinitely for model selection. Not a Kaggle submission.\",\n        \"constraints\":[\"Row-random splitting; repeated prompts may cross splits.\",\"Character lengths are not measures of style or warmth.\",\"Selected C tuned on inner fold only.\",\"No text features: performance reflects length correlates, not causal effects.\"],\n        \"environment\":{\n            \"scikit_learn\":sklearn.__version__,\n            \"pandas\":pd.__version__,\n            \"github_sha\":os.getenv(\"GITHUB_SHA\",\"local\"),\n            \"run_utc\":datetime.now(timezone.utc).isoformat(),\n        }\n    }\n    (output/\"summary.json\").write_text(\n        json.dumps(metrics,indent=2)+\"\\n\",encoding=\"utf-8\"\n    )\n    print(\"Official training rows:\",metrics[\"official_training_rows\"])\n    print(\"Pilot training rows:\",metrics[\"pilot_rows\"])\n    print(\"Chosen inner-fold regularization C:\",best_c)\n    print(\"Outer length-only log loss:\",metrics[\"outer_validation_log_loss_length_only\"])\n    print(\"Outer train-prior log loss:\",metrics[\"outer_validation_log_loss_training_prior\"])\n    print(\"95% bootstrap (length minus prior):\",metrics[\"length_minus_prior_log_loss_bootstrap_95pct\"])\n    print(\"Aggregate summary saved.\")\n    return metrics\n\n\nif __name__ == \"__main__\":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\",default=\"data/train.csv\")\n    p.add_argument(\"--out\",default=\"artifacts/official_length_pilot\")\n    p.add_argument(\"--sample-size\",type=int,default=12000)\n    a=p.parse_args()\n    run_pilot(a.train,a.out,a.sample_size)\n", encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
from src.baseline import TARGETS, get_labels
from src.length_baseline import fit_length_model, predict_length_model


In [ ]:
train = pd.read_csv(root / 'train.csv')
test = pd.read_csv(root / 'test.csv')
assert 'id' in test.columns, 'Kaggle test file must have id column'
targets = get_labels(train)
print('Training examples:', len(train), 'Test examples:', len(test))
# C=10 was chosen on a train-only inner fold in the prior exploratory pilot.
model = fit_length_model(train, targets, c=10.0)
prob = predict_length_model(model, test)
assert prob.shape == (len(test), 3)
assert np.isfinite(prob).all() and np.allclose(prob.sum(axis=1), 1.0)
submission = pd.DataFrame(prob, columns=TARGETS)
submission.insert(0, 'id', test['id'])
destination = Path('/kaggle/working/submission.csv')
submission.to_csv(destination, index=False)
print('Created', destination, 'with', len(submission), 'rows')
print('Columns:', submission.columns.tolist())
